# Sales Analysis — PostgreSQL Gold validation (V1)

**Project:** Pellet Supply Chain Management Dashboard  
**Scope:** Connection and January 2023 sales KPI sanity check against PostgreSQL Gold.  
**Inputs:** `gold.fact_sales`, `gold.fact_sales_budget`; metadata from `information_schema`.  
**Output:** actual tonnes, budget tonnes, variance tonnes and budget attainment percentage.

This notebook is read-only. The SQL statements perform the aggregations; pandas displays their results. This is **not** yet a separate independent calculation in pandas. Source data is synthetic and follows a Directo-inspired model.

## 1. Import dependencies

Uses pandas to display query results, SQLAlchemy and psycopg for connectivity, and `getpass` to prompt for the password without storing it in the notebook.


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from getpass import getpass

## 2. Create a PostgreSQL connection

Defaults: `localhost:5432`, database `postgres`, user `postgres`. Supply your own password at the prompt; do not save passwords in this notebook or commit them to GitHub.


In [3]:
db_password = getpass("PostgreSQL password: ")

engine = create_engine(
    URL.create(
        "postgresql+psycopg",
        username="postgres",
        password=db_password,
        host="localhost",
        port=5432,
        database="postgres"
    )
)

## 3. Verify connectivity

`SELECT 1 AS test` should return `1`. This verifies query execution, not business-data correctness.


In [4]:
pd.read_sql("SELECT 1 AS test", engine)

,test
0,1


## 4. Inspect Gold tables

Lists tables in the `gold` schema to confirm which objects are available. The stored output reflects the database at the time of the previous notebook run.


In [5]:
pd.read_sql("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'gold'
ORDER BY table_name;
""", engine)

,table_schema,table_name
0,gold,dim_channel
1,gold,dim_customer
2,gold,dim_date
3,gold,dim_plant
4,gold,dim_product
5,gold,dim_raw
6,gold,dim_supplier
7,gold,dim_version
8,gold,fact_production
9,gold,fact_production_budget


## 5. Inspect the relevant Gold schema

Shows the column names and data types for `fact_sales`, `fact_sales_budget`, and `dim_date`, supporting validation of the SQL field references.


In [6]:
pd.read_sql("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'gold'
  AND table_name IN ('fact_sales', 'fact_sales_budget', 'dim_date')
ORDER BY table_name, ordinal_position;
""", engine)

,table_name,column_name,data_type
0,dim_date,date,date
1,dim_date,year,integer
2,dim_date,quarter,integer
3,dim_date,month_no,integer
4,dim_date,month_name,text
5,dim_date,year_month,text
6,dim_date,month_start_date,date
7,dim_date,day_of_month,integer
8,dim_date,iso_week,integer
9,dim_date,iso_day_of_week,integer


## 6. January 2023 sales KPI check

The SQL query aggregates all January 2023 sales and budget rows (2023-01-01 inclusive; 2023-02-01 exclusive). No plant, product or channel filter is applied.

- **Actual (t):** `SUM(gold.fact_sales.quantity_t)`
- **Budget (t):** `SUM(gold.fact_sales_budget.budget_t)`
- **Variance (t):** Actual − Budget
- **Attainment (%):** Actual ÷ Budget × 100 (`NULLIF` prevents division by zero)

**Recorded output:** Actual **68,298.7 t**, Budget **68,646.1 t**, Variance **−347.4 t**, Attainment **99.493926%** (approximately **99.5%**). Compare with `sql/01_sales_v1_checks.sql` and Power BI using the same period and filters. Stored output is historical, not evidence of a fresh re-run.


In [7]:
qa_sales_2023_01 = pd.read_sql("""
WITH actual AS (
    SELECT
        SUM(quantity_t) AS actual_t
    FROM gold.fact_sales
    WHERE sales_date >= DATE '2023-01-01'
      AND sales_date <  DATE '2023-02-01'
),
budget AS (
    SELECT
        SUM(budget_t) AS budget_t
    FROM gold.fact_sales_budget
    WHERE budget_date >= DATE '2023-01-01'
      AND budget_date <  DATE '2023-02-01'
)
SELECT
    actual_t,
    budget_t,
    actual_t - budget_t AS variance_t,
    actual_t / NULLIF(budget_t, 0) * 100 AS attainment_pct
FROM actual
CROSS JOIN budget;
""", engine)

qa_sales_2023_01

,actual_t,budget_t,variance_t,attainment_pct
0,68298.7,68646.1,-347.4,99.493926


## Interpretation and limitations

The January 2023 sales actual was 347.4 tonnes below budget; attainment was approximately 99.5%. These figures alone do not explain the variance's business cause.

- Results depend on the current Gold tables; rerun **all** cells to refresh them before presenting new evidence.
- The notebook tests one month, not every period/plant/product/channel combination.
- KPI arithmetic is computed inside PostgreSQL SQL, not recalculated independently in pandas.
- Reconciliations for the full 2023–2025 period and source-to-Bronze QA are documented separately in the project QA files.
- No modification of PostgreSQL data occurs in this notebook.
